# 18 – End-to-End Scenarios

Realistic business user scenarios exercising the full pipeline.  
Each scenario mirrors the intent routing table from the README.

| Scenario | Intent | Agents |
|----------|--------|--------|
| Why did retention drop? | full_diagnostic | All 4 read agents |
| What is the DQ score for CAC? | data_quality | Metadata + Information |
| Who owns the bookings dataset? | governance | Metadata + Knowledge |
| Open Jira bugs for retention | incident_review | Capacity |
| What is GRR? | knowledge_lookup | Knowledge + Metadata |
| Create a bug ticket | write_ticket | Capacity |
| Create a DQ rule | write_rule | Rule |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from graph.graph import build_graph
from graph.state import initial_state

graph = build_graph()

def run(query, **kwargs):
    state = initial_state(query=query, **kwargs)
    result = graph.invoke(state)
    print(f'Query  : {query}')
    print(f'Intent : {result.get("intent")}')
    print(f'Agents : {result.get("_agents_ran")}')
    print(f'Errors : {result.get("errors")}')
    print(f'Summary: {result.get("final_summary", "")[:200]}')
    print()
    return result

## Scenario 1 — Full diagnostic (all 4 read agents)

In [ ]:
r = run(
    'Give me a full diagnostic overview of all data products',
    data_products=['retention', 'bookings', 'cac', 'ltv'],
)

## Scenario 2 — Data quality check

In [ ]:
r = run('What is the DQ score for the CAC dataset?', data_products=['cac'])

## Scenario 3 — Governance / ownership lookup

In [ ]:
r = run('Who owns the bookings dataset and what is the governance policy?')

## Scenario 4 — Incident review (Jira)

In [ ]:
r = run('Show me open Jira incidents for retention')

## Scenario 5 — Knowledge lookup

In [ ]:
r = run('What is GRR and how is it calculated?')

## Scenario 6 — Write: Create Jira ticket

In [ ]:
r = run(
    'Create ticket: retention pipeline has been down for 3 hours in EU region',
    data_products=['retention'],
)

for ar in r.get('agent_results', []):
    if ar['agent'] == 'capacity':
        data = ar.get('data') or {}
        ticket_id = data.get('ticket_id')
        if ticket_id:
            print(f'Ticket created: {ticket_id}')

## Scenario 7 — Write: Create DQ rule

In [ ]:
r = run(
    'Create a new data quality rule to validate LTV completeness',
    data_products=['ltv'],
)

for ar in r.get('agent_results', []):
    if ar['agent'] == 'rule':
        data = ar.get('data') or {}
        if isinstance(data, dict) and 'id' in data:
            print(f'Rule created: {data["id"]} — {data.get("name")}')

## Scenario 8 — Blocked by guardrail

In [ ]:
r = run('DROP TABLE analytics.retention_metrics')
print('guardrail_passed:', r.get('guardrail_passed'))
print('final_summary   :', r.get('final_summary'))